<a href="https://colab.research.google.com/github/EduardoAve/Labour-well-being/blob/main/models/model_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
pip install semopy

In [8]:
import pandas as pd
import semopy
import scipy.stats
import numpy as np

In [9]:
df = pd.read_csv('https://raw.githubusercontent.com/EduardoAve/Labour-well-being/refs/heads/main/data/02_prepared/Final_Dataset_Processed-2.csv')


#Academic = 2

In [10]:
df_CZ = df[df['Country'] == 2]
df_CZ_Academic = df_CZ[df_CZ['Academic/Non-academic'] == 2]
df_CZ_Academic.info()

<class 'pandas.core.frame.DataFrame'>
Index: 897 entries, 0 to 952
Data columns (total 59 columns):
 #   Column                                                       Non-Null Count  Dtype  
---  ------                                                       --------------  -----  
 0   Country                                                      897 non-null    int64  
 1   Version                                                      897 non-null    int64  
 2   Gender                                                       897 non-null    float64
 3   Age                                                          897 non-null    float64
 4   Marital_Status                                               897 non-null    float64
 5   Cares_for_Dependents                                         897 non-null    float64
 6   Institution_Type                                             897 non-null    float64
 7   Subject_Area                                                 897 non-null    float64


In [11]:
import pandas as pd
import numpy as np
import semopy
import scipy.stats # Para la prueba de Sobel
# import seaborn as sns # Descomenta si quieres ver heatmaps
# import matplotlib.pyplot as plt # Descomenta si quieres ver heatmaps

# ---------------------------------------------------------------------------
# PASO 0: Cargar tus Datos desde la URL y Renombrar Columnas
# ---------------------------------------------------------------------------
data_url = "https://raw.githubusercontent.com/EduardoAve/Labour-well-being/refs/heads/main/data/02_prepared/labour_wellbeing_processed_v1.csv"
try:
    df = pd.read_csv(data_url)
    print(f"Archivo CSV cargado exitosamente desde: {data_url}")
except Exception as e:
    print(f"Ocurrió un error al cargar el CSV desde la URL: {e}")
    exit()

try:
    df.rename(columns={'Effort [%]': 'Effort_perc', 'Income EURO': 'Income_EURO'}, inplace=True)
    print("Columnas renombradas (si existían).")
except KeyError:
    print("Advertencia: Alguna columna a renombrar no estaba presente.")
print("-" * 50)

# ---------------------------------------------------------------------------
# PASO 1: Creación de Variables de Motivación Compuestas
# ---------------------------------------------------------------------------
motivation_cols_for_controlled = ['WM_Extrinsic_Social', 'WM_Extrinsic_Material', 'WM_Introjected_Motivation']
motivation_cols_for_autonomous = ['WM_Identified_Motivation', 'WM_Intrinsic_Motivation']
df['WM_Controlled_Motivation'] = df[motivation_cols_for_controlled].mean(axis=1)
df['WM_Autonomous_Motivation'] = df[motivation_cols_for_autonomous].mean(axis=1)
print("Variables de motivación compuestas creadas.")
print("-" * 50)

# ---------------------------------------------------------------------------
# Lista de Indicadores de Condiciones de Trabajo (ahora predictores observados)
# ---------------------------------------------------------------------------
indicadores_X_observados = [
    'Avg_Work_Hours_HE', 'Income_EURO', 'Effort_perc', 'Policy_Influence',
    'Academic_Resources', 'Performance_Pressure', 'Perceived_Autonomy',
    'Quality_Leadership', 'Sense_Community'
]

# ---------------------------------------------------------------------------
# DIAGNÓSTICO DE DATOS
# ---------------------------------------------------------------------------
columnas_del_modelo_actual = [
    'WM_Controlled_Motivation', 'WM_Autonomous_Motivation',
    'Vulnerability', 'Burnout_Score'
] + indicadores_X_observados # Añadir los 9 indicadores

missing_cols_in_df = [col for col in columnas_del_modelo_actual if col not in df.columns]
if missing_cols_in_df:
    print(f"Error CRÍTICO: Faltan columnas necesarias para el modelo en el DataFrame: {missing_cols_in_df}")
    exit()

print("Verificando NaNs en las columnas finales del modelo:")
nan_counts = df[columnas_del_modelo_actual].isnull().sum()
print(nan_counts)
total_nans_in_model_cols = nan_counts.sum()
print(f"Total de NaNs en estas columnas del modelo: {total_nans_in_model_cols}")
if total_nans_in_model_cols > 0:
    print("ADVERTENCIA: Hay datos faltantes. Se intentará usar FIML.")
print("-" * 50)

# ---------------------------------------------------------------------------
# PASO 2: Especificación del Modelo de Senderos (Path Analysis)
# ---------------------------------------------------------------------------
# Crear la cadena de predictores para las condiciones de trabajo
predictores_X_str = " + ".join(indicadores_X_observados)

model_spec_path_analysis = f"""
# Modelo Estructural con Indicadores de X como predictores observados separados

# Influencia de los 9 indicadores de Condiciones de Trabajo sobre Motivaciones
WM_Controlled_Motivation ~ {predictores_X_str}
WM_Autonomous_Motivation ~ {predictores_X_str}

# Influencia de Motivaciones sobre Vulnerabilidad
Vulnerability ~ WM_Controlled_Motivation + WM_Autonomous_Motivation

# Influencia de los 9 indicadores de Condiciones de Trabajo y Vulnerabilidad sobre Burnout
Burnout_Score ~ {predictores_X_str} + Vulnerability
"""
print("Especificación del Modelo de Senderos:")
print(model_spec_path_analysis)
print("-" * 50)

# ---------------------------------------------------------------------------
# PASO 3: Creación y Ajuste del Modelo
# ---------------------------------------------------------------------------
model = semopy.Model(model_spec_path_analysis)
results_optimization = None
converged_based_on_solver = False

print("Intentando ajuste del modelo de senderos...")
estimator_to_use = 'MLW'
df_for_model = df[columnas_del_modelo_actual].copy()

if total_nans_in_model_cols > 0:
    print(f"ADVERTENCIA: Se detectaron {total_nans_in_model_cols} NaNs. Intentando usar FIML.")
    estimator_to_use = 'FIML'
else:
    print(f"No se detectaron NaNs en las columnas del modelo. Usando estimador {estimator_to_use}.")
    initial_rows = len(df_for_model)
    df_for_model.dropna(inplace=True)
    if len(df_for_model) < initial_rows:
         print(f"Se eliminaron {initial_rows - len(df_for_model)} filas con NaNs de df_for_model para el estimador MLW.")

try:
    results_optimization = model.fit(data=df_for_model, obj=estimator_to_use)
    print(f"\nInformación directa de 'results_optimization' (optimizador {estimator_to_use}):")
    if hasattr(results_optimization, 'fun'): print(f"  Objective value (fun): {results_optimization.fun}")
    if hasattr(results_optimization, 'success'): print(f"  Success flag: {results_optimization.success}")
    if hasattr(results_optimization, 'status'): print(f"  Status code: {results_optimization.status}")
    if hasattr(results_optimization, 'message'): print(f"  Message: {results_optimization.message}")
    if hasattr(results_optimization, 'nit'): print(f"  Iterations (nit): {results_optimization.nit}")

    if hasattr(results_optimization, 'success') and results_optimization.success:
        converged_based_on_solver = True
        print("  Interpretación: El optimizador reportó ÉXITO.")
    elif hasattr(results_optimization, 'status') and results_optimization.status == 0:
        converged_based_on_solver = True
        print("  Interpretación: El optimizador reportó un CÓDIGO DE ESTADO de 0 (típicamente éxito).")
    else:
        print("  Interpretación: El optimizador NO reportó éxito explícitamente o un código de estado conocido de éxito.")
    print("-" * 50)

except Exception as e:
    print(f"Error CRÍTICO durante el ajuste del modelo con {estimator_to_use}: {e}")
    print("-" * 50)

# ---------------------------------------------------------------------------
# PASO 4: Evaluación de la Bondad de Ajuste del Modelo
# ---------------------------------------------------------------------------
if converged_based_on_solver:
    print("Intentando calcular índices de ajuste...")
    try:
        fit_indices = semopy.calc_stats(model)
        print("\nÍndices de Bondad de Ajuste:")
        print(fit_indices.T)
    except Exception as e:
        print(f"Error al calcular los índices de ajuste: {e}")
    print("-" * 50)
else:
    print("La optimización del modelo no reportó éxito o falló. No se calcularán índices de ajuste.")
    print("-" * 50)

# ---------------------------------------------------------------------------
# PASO 5: Interpretación de los Coeficientes de las Rutas Directas
# ---------------------------------------------------------------------------
estimates_df = None
if converged_based_on_solver:
    print("Intentando inspeccionar parámetros...")
    try:
        estimates_df = model.inspect()
        if estimates_df is not None and not estimates_df.empty:
            print("\nEstimaciones de los Parámetros del Modelo (No Estandarizadas):")
            print(estimates_df)
            # No hay "varianzas de error de indicadores" en el mismo sentido que con X latente,
            # pero sí habrá varianzas residuales para las variables endógenas.
            # Puedes buscar si alguna varianza residual de las variables endógenas es negativa,
            # aunque es menos común en modelos de path analysis solo con observadas si las variables son razonables.
            if 'Std. Err' in estimates_df.columns:
                 residual_variances = estimates_df[estimates_df['op'] == '~~']
                 self_variances = residual_variances[residual_variances['lval'] == residual_variances['rval']]
                 # Filtra las variables exógenas (los 9 indicadores de X) de este chequeo, ya que sus varianzas son datos, no estimaciones de error.
                 endogenous_obs_vars = ['WM_Controlled_Motivation', 'WM_Autonomous_Motivation', 'Vulnerability', 'Burnout_Score']
                 problematic_variances = self_variances[self_variances['lval'].isin(endogenous_obs_vars) & (self_variances['Estimate'] < 0)]
                 if not problematic_variances.empty:
                     print("\nADVERTENCIA: Se detectaron varianzas residuales negativas para variables endógenas:")
                     print(problematic_variances)
        else:
            print("\nADVERTENCIA: model.inspect() devolvió vacío o None.")
            estimates_df = None
        print("-" * 50)
    except Exception as e:
        print(f"Error al inspeccionar los resultados del modelo: {e}")
        estimates_df = None
        print("-" * 50)
else:
    print("La optimización del modelo no reportó éxito o falló. Los parámetros no se mostrarán.")
    print("-" * 50)

# ---------------------------------------------------------------------------
# PASO 6: Cálculo y Prueba de Significancia de los Efectos Indirectos
# ---------------------------------------------------------------------------
# Calcular todos los efectos indirectos ahora es más complejo porque X se descompuso.
# Mostraremos un ejemplo para UNA de las antiguas variables X a través de UNA vía de motivación.
# Deberás replicar esto para otros indicadores de X y la otra vía de motivación si es necesario.

if estimates_df is not None and not estimates_df.empty and 'Std. Err' in estimates_df.columns:
    print("\nCalculando Ejemplo de Efecto Indirecto (Prueba de Sobel)...")
    print("Ejemplo para: Avg_Work_Hours_HE -> WM_Controlled_Motivation -> Vulnerability -> Burnout_Score")

    try:
        # Ruta a1: Indicador_X_especifico -> WM_Controlled_Motivation
        path_IndX_WM_C = estimates_df.loc[
            (estimates_df['lval'] == 'WM_Controlled_Motivation') &
            (estimates_df['rval'] == 'Avg_Work_Hours_HE') # CAMBIA 'Avg_Work_Hours_HE' por otro indicador si quieres
        ]
        # Ruta b1: WM_Controlled_Motivation -> Vulnerability
        path_WM_C_V = estimates_df.loc[
            (estimates_df['lval'] == 'Vulnerability') &
            (estimates_df['rval'] == 'WM_Controlled_Motivation')
        ]
        # Ruta c1: Vulnerability -> Burnout_Score
        path_V_BO = estimates_df.loc[
            (estimates_df['lval'] == 'Burnout_Score') &
            (estimates_df['rval'] == 'Vulnerability')
        ]

        if not (path_IndX_WM_C.empty or path_WM_C_V.empty or path_V_BO.empty):
            if path_IndX_WM_C['Std. Err'].isnull().any() or \
               path_WM_C_V['Std. Err'].isnull().any() or \
               path_V_BO['Std. Err'].isnull().any():
                print("  No se puede calcular este efecto indirecto debido a Std. Err faltantes en alguna ruta.")
            else:
                a1, se_a1 = path_IndX_WM_C['Estimate'].iloc[0], path_IndX_WM_C['Std. Err'].iloc[0]
                b1, se_b1 = path_WM_C_V['Estimate'].iloc[0], path_WM_C_V['Std. Err'].iloc[0]
                c1, se_c1 = path_V_BO['Estimate'].iloc[0], path_V_BO['Std. Err'].iloc[0]

                IE1 = a1 * b1 * c1
                SE_IE1_sq = (a1**2*b1**2*se_c1**2) + (a1**2*c1**2*se_b1**2) + (b1**2*c1**2*se_a1**2)
                print(f"\nEfecto Indirecto (Avg_Work_Hours_HE -> WM_C -> V -> BO): {IE1}") # Ajusta el nombre
                if SE_IE1_sq > 0 and not np.isnan(SE_IE1_sq) and se_a1 > 0 and se_b1 > 0 and se_c1 > 0 :
                    SE_IE1 = np.sqrt(SE_IE1_sq)
                    if SE_IE1 > 1e-9:
                        z_IE1 = IE1 / SE_IE1
                        p_IE1 = 2 * (1 - scipy.stats.norm.cdf(abs(z_IE1)))
                        print(f"  SE(Sobel): {SE_IE1:.4f}, z: {z_IE1:.2f}, p: {p_IE1:.4f}")
                    else: print(f"  SE(Sobel) {SE_IE1:.4f} cercano a cero.")
                else: print(f"  No se calcula SE(Sobel) (varianza_sq: {SE_IE1_sq:.4f} o SEs no positivos).")
        else:
            print("  Faltan rutas para calcular el ejemplo de efecto indirecto.")
        print("-" * 50)

    except KeyError as e:
        print(f"Error (KeyError) calculando ejemplo de efecto indirecto: {e}.")
    except Exception as e:
        print(f"Error calculando ejemplo de efecto indirecto: {e}")
else:
    print("Estimaciones no disponibles/vacías o el modelo no convergió adecuadamente. No se calculan efectos indirectos.")

print("\nAnálisis SEM (modelo de senderos) finalizado.")

Archivo CSV cargado exitosamente desde: https://raw.githubusercontent.com/EduardoAve/Labour-well-being/refs/heads/main/data/02_prepared/labour_wellbeing_processed_v1.csv
Columnas renombradas (si existían).
--------------------------------------------------
Variables de motivación compuestas creadas.
--------------------------------------------------
Verificando NaNs en las columnas finales del modelo:
WM_Controlled_Motivation    0
WM_Autonomous_Motivation    0
Vulnerability               0
Burnout_Score               0
Avg_Work_Hours_HE           0
Income_EURO                 0
Effort_perc                 0
Policy_Influence            0
Academic_Resources          0
Performance_Pressure        0
Perceived_Autonomy          0
Quality_Leadership          0
Sense_Community             0
dtype: int64
Total de NaNs en estas columnas del modelo: 0
--------------------------------------------------
Especificación del Modelo de Senderos:

# Modelo Estructural con Indicadores de X como predicto

In [12]:
try:
    # Obtener las estimaciones estandarizadas
    estimates_df_std = model.inspect(std_est=True)

    if estimates_df_std is not None and not estimates_df_std.empty:
        print("\nEstimaciones de los Parámetros del Modelo (ESTANDARIZADAS):")
        print(estimates_df_std)
    else:
        print("\nNo se pudieron calcular u obtener las estimaciones estandarizadas.")
        print("Asegúrate de que el modelo haya convergido y que 'model.inspect(std_est=True)' funcione.")

except Exception as e:
    print(f"Ocurrió un error al intentar obtener los coeficientes estandarizados: {e}")
    print("Asegúrate de que el objeto 'model' esté correctamente ajustado.")



Estimaciones de los Parámetros del Modelo (ESTANDARIZADAS):
                        lval  op                      rval  Estimate  \
0   WM_Controlled_Motivation   ~         Avg_Work_Hours_HE  0.004724   
1   WM_Controlled_Motivation   ~               Income_EURO -0.000045   
2   WM_Controlled_Motivation   ~               Effort_perc -0.001462   
3   WM_Controlled_Motivation   ~          Policy_Influence -0.074556   
4   WM_Controlled_Motivation   ~        Academic_Resources  0.143792   
5   WM_Controlled_Motivation   ~      Performance_Pressure  0.170813   
6   WM_Controlled_Motivation   ~        Perceived_Autonomy -0.143039   
7   WM_Controlled_Motivation   ~        Quality_Leadership  0.048961   
8   WM_Controlled_Motivation   ~           Sense_Community  0.061942   
9   WM_Autonomous_Motivation   ~         Avg_Work_Hours_HE  0.009287   
10  WM_Autonomous_Motivation   ~               Income_EURO  0.000018   
11  WM_Autonomous_Motivation   ~               Effort_perc  0.002854   
12 